In [ ]:
import os
import pandas as pd

B_DIR = "" # Enter path of the folder with .b files present in it
WORD_DIR = "" # Enter path of the folder with .wrd files present in it
OUTPUT_DIR = "/home/speechlab/Desktop/Meity_Tamil/tamil_wrd_with_silence" # enter path of the output dir needed , this is a example

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIME_TOLERANCE = 0.02   # 20 ms tolerance for timing mismatch

def get_boundary_silence(word_end, next_word_start, b_df, tol=TIME_TOLERANCE):
    """
    Returns LSIL / MSIL / SSIL if a silence overlaps the word boundary
    within a tolerance window. Otherwise returns NSIL.
    """

    silences = b_df[
        (b_df["Endtime"] > word_end - tol) &
        (b_df["Starttime"] < next_word_start + tol)
    ]

    if not silences.empty:
        # Choose the longest overlapping silence
        silences = silences.copy()
        silences["duration"] = silences["Endtime"] - silences["Starttime"]
        return silences.sort_values("duration", ascending=False).iloc[0]["Break_Index"]

    return "NSIL"

for filename in os.listdir(B_DIR):
    if not filename.endswith(".b"):
        continue

    b_path = os.path.join(B_DIR, filename)
    word_path = os.path.join(WORD_DIR, filename.replace(".b", ".wrd"))

    # Skip if corresponding .word file doesn't exist
    if not os.path.exists(word_path):
        print(f"⚠️ Skipping {filename}: matching .word file not found")
        continue

    print(f"Processing: {filename}")

    word_df = pd.read_csv(word_path, sep=r"\s+", engine="python")
    b_df = pd.read_csv(b_path, sep=r"\s+", engine="python")

    word_df.columns = ["Starttime", "Endtime", "Word"]
    b_df.columns = ["Starttime", "Endtime", "Break_Index"]

    # Sort to be safe
    word_df = word_df.sort_values("Starttime").reset_index(drop=True)
    b_df = b_df.sort_values("Starttime").reset_index(drop=True)

    output_rows = []

    for i in range(len(word_df)):
        # Add original word row
        output_rows.append({
            "Starttime": word_df.loc[i, "Starttime"],
            "Endtime": word_df.loc[i, "Endtime"],
            "Token": word_df.loc[i, "Word"]
        })

        # Add silence row (except after last word)
        if i < len(word_df) - 1:
            word_end = word_df.loc[i, "Endtime"]
            next_word_start = word_df.loc[i + 1, "Starttime"]

            silence_label = get_boundary_silence(
                word_end,
                next_word_start,
                b_df
            )

            output_rows.append({
                "Starttime": word_end,
                "Endtime": next_word_start,
                "Token": silence_label
            })

    output_df = pd.DataFrame(output_rows)

    output_path = os.path.join(
        OUTPUT_DIR,
        filename.replace(".b", ".wrd")
    )

    output_df.to_csv(
        output_path,
        sep=" ",
        index=False,
        header=True
    )

print("\n✅ All files processed successfully with tolerance-based silence alignment.")


Processing: train_tamilmale_06088.b
Processing: train_tamilmale_06524.b
Processing: train_tamilmale_04730.b
Processing: train_tamilmale_05218.b
Processing: train_tamilmale_04911.b
Processing: train_tamilmale_04417.b
Processing: train_tamilmale_06281.b
Processing: train_tamilmale_05405.b
Processing: train_tamilmale_05285.b
Processing: train_tamilmale_06327.b
Processing: train_tamilmale_05913.b
Processing: train_tamilmale_03898.b
Processing: train_tamilmale_06282.b
Processing: train_tamilmale_06273.b
Processing: train_tamilmale_03963.b
Processing: train_tamilmale_05072.b
Processing: train_tamilmale_04562.b
Processing: train_tamilmale_04436.b
Processing: train_tamilmale_05359.b
Processing: train_tamilmale_03920.b
Processing: train_tamilmale_04392.b
Processing: train_tamilmale_05004.b
Processing: train_tamilmale_06092.b
Processing: train_tamilmale_05632.b
Processing: train_tamilmale_03224.b
Processing: train_tamilmale_03127.b
Processing: train_tamilmale_03510.b
Processing: train_tamilmale_

In [10]:
import os

WRD_DIR = "/home/speechlab/Desktop/Meity_Tamil/tamil_wrd_with_silence" #example path
OUTPUT_TEXT_FILE = "/home/speechlab/Desktop/Meity_Tamil/final_sentences_with_silence.txt" # to add silence

# =====================================================
# PROCESS ALL .wrd FILES
# =====================================================
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as out_f:

    for filename in sorted(os.listdir(WRD_DIR)):
        if not filename.endswith(".wrd"):
            continue

        file_path = os.path.join(WRD_DIR, filename)

        # Label from filename
        label = filename.replace(".wrd", "")

        tokens = []

        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

            # Skip header
            for line in lines[1:]:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue

                token = parts[2]
                tokens.append(token)

        # Write label
        out_f.write(label + "\n")

        # Write full sentence
        sentence = " ".join(tokens)
        out_f.write(sentence + "\n\n")

print("✅ All sentences written into one file successfully.")


✅ All sentences written into one file successfully.


In [12]:
import os
import re

# =====================================================
# PATHS
# =====================================================
TRANS_FILE = "/home/speechlab/Desktop/Meity_Tamil/final_sentences_with_silence.txt"
#the output in word dir will be in englishh , we have to turn it into tamil , and hence this block of code 
TAMIL_TEXT_FILE = "/home/speechlab/Desktop/Meity_Tamil/audio and tamil_text/Tamil_male_mono.txt"
OUTPUT_FILE = "/home/speechlab/Desktop/Meity_Tamil/final_sentences_tamil_with_silence.txt" #final tamil file with silence information 

SILENCE_TOKENS = {"NSIL", "LSIL", "MSIL", "SSIL"}

tamil_dict = {}

with open(TAMIL_TEXT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            continue

        label, sentence = parts

        # Remove punctuation, keep only Tamil words
        sentence = re.sub(r"[^\u0B80-\u0BFF\s]", "", sentence)

        tamil_words = sentence.split()
        tamil_dict[label] = tamil_words

with open(TRANS_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    lines = [line.rstrip() for line in fin]

    i = 0
    while i < len(lines):
        label = lines[i]
        sentence = lines[i + 1]
        i += 2

        tokens = sentence.split()

        if label not in tamil_dict:
            print(f"⚠️ Tamil text not found for {label}, skipping")
            continue

        tamil_words = tamil_dict[label]
        tamil_idx = 0
        output_tokens = []

        for tok in tokens:
            if tok in SILENCE_TOKENS:
                output_tokens.append(tok)
            else:
                if tamil_idx < len(tamil_words):
                    output_tokens.append(tamil_words[tamil_idx])
                    tamil_idx += 1
                else:
                    output_tokens.append(tok)  # fallback safety

        fout.write(label + "\n")
        fout.write(" ".join(output_tokens) + "\n\n")

print("✅ Tamil words replaced successfully, silences preserved.")


⚠️ Tamil text not found for , skipping
⚠️ Tamil text not found for sari NSIL meedxam LSIL keetxpaar NSIL enrxu NSIL ndinaittaal LSIL adarxkum MSIL titxtxu NSIL wizhum, skipping
⚠️ Tamil text not found for , skipping
⚠️ Tamil text not found for mattiya NSIL arasu LSIL tagudiyaana NSIL ndabargalxait MSIL teernddedxuttut LSIL taramaana LSIL ndalla MSIL parandda MSIL kalwi NSIL murxai NSIL wazhangga LSIL ndadxawadxikkai NSIL edxuttulxlxadeu, skipping
⚠️ Tamil text not found for , skipping
⚠️ Tamil text not found for manmadan LSIL radi LSIL uruwa NSIL amaippu LSIL manmadan NSIL iranxdxu LSIL ndaanku NSIL alladu NSIL etxtxuk NSIL karanggalxudxan LSIL kaanxappadxuwaar, skipping
⚠️ Tamil text not found for , skipping
⚠️ Tamil text not found for ikkadaik MSIL kawidaigalxil LSIL kaudaman LSIL agaligaikkum LSIL inddiraneukkum NSIL saabam SSIL tarum SSIL seyal LSIL idxam NSIL perxawillai, skipping
⚠️ Tamil text not found for , skipping
⚠️ Tamil text not found for panjjaap NSIL anxi LSIL munnadaaga